# PyTorch

In [1]:
import os
import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.optim as optim

if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    os.environ["DISPLAY"] = ":1"

env = gym.make("CartPole-v1")
s, _ = env.reset()

n_actions = env.action_space.n
state_dim = env.observation_space.shape[0]

device = torch.device("cpu")

In [2]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.model(x)


network = QNetwork(state_dim, n_actions).to(device)
optimizer = optim.Adam(network.parameters(), lr=1e-4)
gamma = 0.99

Проверка

In [3]:
@torch.no_grad()
def get_action(state, epsilon=0.0):
    if isinstance(state, tuple):
        state = state[0]

    if np.random.rand() < epsilon:
        return int(np.random.randint(n_actions))

    state_v = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    q_values = network(state_v)[0]
    return int(torch.argmax(q_values).item())


s, _ = env.reset()
assert np.shape(get_action(s)) == (), "верните только одно действие (integer)"

for eps in [0.0, 0.1, 0.5, 1.0]:
    state_frequencies = np.bincount(
        [get_action(s, epsilon=eps) for _ in range(10000)],
        minlength=n_actions
    )
    best_action = state_frequencies.argmax()

    assert abs(state_frequencies[best_action] - 10000 * (1 - eps + eps / n_actions)) < 200
    for other_action in range(n_actions):
        if other_action != best_action:
            assert abs(state_frequencies[other_action] - 10000 * (eps / n_actions)) < 200

    print("e=%.1f tests passed" % eps)

e=0.0 tests passed
e=0.1 tests passed
e=0.5 tests passed
e=1.0 tests passed


Обучение

In [4]:
def train_step(state, action, reward, next_state, done):
    state_v = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    next_state_v = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
    action_v = torch.tensor([action], dtype=torch.int64, device=device)
    reward_v = torch.tensor([reward], dtype=torch.float32, device=device)
    done_v = torch.tensor([done], dtype=torch.bool, device=device)

    predicted_qvalues = network(state_v)                     
    predicted_qvalues_for_actions = predicted_qvalues.gather(1, action_v.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        predicted_next_qvalues = network(next_state_v)         
        next_state_values = predicted_next_qvalues.max(dim=1).values
        target_qvalues_for_actions = reward_v + gamma * next_state_values
        target_qvalues_for_actions = torch.where(done_v, reward_v, target_qvalues_for_actions)

    loss = (predicted_qvalues_for_actions - target_qvalues_for_actions) ** 2
    loss = loss.mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

Игры с моделью

In [5]:
def generate_session(env, t_max=1000, epsilon=0.0, train=False):
    total_reward = 0.0
    s, _ = env.reset()

    for _ in range(t_max):
        a = get_action(s, epsilon=epsilon)

        next_s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated

        if train:
            train_step(s, a, r, next_s, done)

        total_reward += r
        s = next_s

        if done:
            break

    return total_reward

epsilon = 0.5
reward_history = []

for i in range(1000):
    session_rewards = [generate_session(env, epsilon=epsilon, train=True) for _ in range(100)]
    mean_reward = np.mean(session_rewards)
    reward_history.append(mean_reward)

    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(i, mean_reward, epsilon))

    epsilon = max(epsilon * 0.99, 1e-4)

    if mean_reward > 300:
        print("You Win!")
        break

epoch #0	mean reward = 15.100	epsilon = 0.500
epoch #1	mean reward = 13.600	epsilon = 0.495
epoch #2	mean reward = 13.780	epsilon = 0.490
epoch #3	mean reward = 15.080	epsilon = 0.485
epoch #4	mean reward = 13.360	epsilon = 0.480
epoch #5	mean reward = 14.390	epsilon = 0.475
epoch #6	mean reward = 13.340	epsilon = 0.471
epoch #7	mean reward = 22.780	epsilon = 0.466
epoch #8	mean reward = 16.820	epsilon = 0.461
epoch #9	mean reward = 25.630	epsilon = 0.457
epoch #10	mean reward = 31.570	epsilon = 0.452
epoch #11	mean reward = 33.760	epsilon = 0.448
epoch #12	mean reward = 34.270	epsilon = 0.443
epoch #13	mean reward = 36.280	epsilon = 0.439
epoch #14	mean reward = 44.830	epsilon = 0.434
epoch #15	mean reward = 51.870	epsilon = 0.430
epoch #16	mean reward = 57.740	epsilon = 0.426
epoch #17	mean reward = 65.200	epsilon = 0.421
epoch #18	mean reward = 77.670	epsilon = 0.417
epoch #19	mean reward = 76.560	epsilon = 0.413
epoch #20	mean reward = 104.820	epsilon = 0.409
epoch #21	mean reward 